# 🚀 Day Trading Strategy - Midnight Entry
**Complete Trading Framework with Delta Exchange Integration**

## 📋 Strategy Overview:
- **Entry**: Midnight (12 AM IST) or custom entry time
- **Exit**: End of day (4 PM IST) or TP/SL triggered
- **Positions**: LONG, SHORT, or BOTH
- **Risk Management**: Configurable TP, SL, Commission, Slippage

## 📊 Blocks Overview:
1. **API Credentials & Authentication** - Delta Exchange Setup
2. **Trading Parameters** - Day Trading Configuration
3. **Technical Indicators** - Strategy Logic
4. **Data Fetching** - Delta Exchange Historical Data
5. **Backtest Engine** - Backtrader Execution & Results
6. **Trade Logs** - TradingView Format Reports with Exit Reasons
7. **Interactive Charts** - Candlestick + Signals (Scrollable)

**Run blocks sequentially from top to bottom ↓**

---
## BLOCK 1️⃣ : API CREDENTIALS & DELTA EXCHANGE AUTHENTICATION

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 1: API CREDENTIALS & DELTA EXCHANGE AUTHENTICATION
# ═══════════════════════════════════════════════════════════════════════════════

import hashlib
import hmac
import requests
import time
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# ═════════════════════════════════════════════════════════════════════════════════
# 🔐 DELTA EXCHANGE API CREDENTIALS
# ═════════════════════════════════════════════════════════════════════════════════

API_KEY = 'enter api key'
API_SECRET = 'enter api secret '
BASE_URL = 'https://api.delta.exchange'
HISTORY_URL = 'https://api.india.delta.exchange/v2/history/candles'

# ═════════════════════════════════════════════════════════════════════════════════
# ✅ DELTA EXCHANGE API VERIFICATION
# ═════════════════════════════════════════════════════════════════════════════════

def generate_signature(secret, message):
    """Generate HMAC-SHA256 signature for API authentication"""
    message = bytes(message, 'utf-8')
    secret = bytes(secret, 'utf-8')
    hash_obj = hmac.new(secret, message, hashlib.sha256)
    return hash_obj.hexdigest()

# Verify API credentials format
test_message = "test_authentication"
test_signature = generate_signature(API_SECRET, test_message)

print("╔" + "═"*80 + "╗")
print("║" + "✅ DELTA EXCHANGE API CREDENTIALS VERIFICATION".center(80) + "║")
print("╚" + "═"*80 + "╝\n")

print(f"🔐 API Key:        {API_KEY[:15]}...{API_KEY[-5:]}")
print(f"🔑 API Secret:     {API_SECRET[:10]}...{API_SECRET[-5:]}")
print(f"🌐 Base URL:       {BASE_URL}")
print(f"📡 History URL:    {HISTORY_URL}")
print(f"\n✓ Signature Algorithm: HMAC-SHA256")
print(f"✓ Test Signature:  {test_signature[:20]}...")
print(f"\n⏰ Current Time UTC: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✅ Authentication Ready!\n")

╔════════════════════════════════════════════════════════════════════════════════╗
║                 ✅ DELTA EXCHANGE API CREDENTIALS VERIFICATION                  ║
╚════════════════════════════════════════════════════════════════════════════════╝

🔐 API Key:        enter api key...i key
🔑 API Secret:     enter api ...cret 
🌐 Base URL:       https://api.delta.exchange
📡 History URL:    https://api.india.delta.exchange/v2/history/candles

✓ Signature Algorithm: HMAC-SHA256
✓ Test Signature:  69f859f2ddfa45e96737...

⏰ Current Time UTC: 2026-05-11 05:32:50
✅ Authentication Ready!



---
## BLOCK 2️⃣ : TRADING PARAMETERS & STRATEGY CONFIGURATION

In [26]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 2: TRADING PARAMETERS & STRATEGY CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("⚙️  DAY TRADING STRATEGY PARAMETERS".center(80))
print("="*80 + "\n")

# ═════════════════════════════════════════════════════════════════════════════════
# 🎯 TRADING MODE SELECTION
# ═════════════════════════════════════════════════════════════════════════════════

TRADING_MODE = "LONG_ONLY"  # Options: "LONG_ONLY", "SHORT_ONLY", "BOTH"

# Set LONG_ENABLED and SHORT_ENABLED based on TRADING_MODE
if TRADING_MODE == "LONG_ONLY":
    LONG_ENABLED = True
    SHORT_ENABLED = False
elif TRADING_MODE == "SHORT_ONLY":
    LONG_ENABLED = False
    SHORT_ENABLED = True
else:  # BOTH
    LONG_ENABLED = True
    SHORT_ENABLED = True

# ═════════════════════════════════════════════════════════════════════════════════
# ⏰ ENTRY TIME CONFIGURATION (IST - Indian Standard Time)
# ═════════════════════════════════════════════════════════════════════════════════

ENTRY_HOUR = 10                   # Hour of entry (0-23 in IST)
ENTRY_MIN = 30                     # Minute of entry (0-59)
ENTRY_TIME_IN_MIN = ENTRY_HOUR * 60 + ENTRY_MIN  # Convert to minutes from midnight

# ═════════════════════════════════════════════════════════════════════════════════
# 🚪 EXIT TIME & CONDITIONS
# ═════════════════════════════════════════════════════════════════════════════════

EXIT_HOUR = 23                    # End of day hour (23:58 IST = 11:58 PM)
EXIT_MIN = 58                      # End of day minute
EXIT_TIME_IN_MIN = EXIT_HOUR * 60 + EXIT_MIN  # Convert to minutes from midnight

# ═════════════════════════════════════════════════════════════════════════════════
# 🎯 TAKE PROFIT CONFIGURATION (Percentage OR Points)
# ═════════════════════════════════════════════════════════════════════════════════

USE_TAKE_PROFIT = True            # Enable take profit
TP_MODE = "POINTS"            # Options: "PERCENTAGE", "POINTS", "BOTH"

# Percentage-based TP
TAKE_PROFIT_PERCENT = 0.4        # Take profit percentage (%)

# Points-based TP (Dollar amount)
TAKE_PROFIT_POINTS = 350      # Take profit in USD per trade

# ═════════════════════════════════════════════════════════════════════════════════
# 🛑 STOP LOSS CONFIGURATION (Percentage OR Points)
# ═════════════════════════════════════════════════════════════════════════════════

USE_STOP_LOSS = True              # Enable stop loss
SL_MODE = "POINTS"            # Options: "PERCENTAGE", "POINTS", "BOTH"

# Percentage-based SL
STOP_LOSS_PERCENT = 4           # Stop loss percentage (%)

# Points-based SL (Dollar amount)
STOP_LOSS_POINTS = 10000          # Stop loss in USD per trade

# ═════════════════════════════════════════════════════════════════════════════════
# 💰 POSITION SIZING & RISK MANAGEMENT
# ═════════════════════════════════════════════════════════════════════════════════

INITIAL_CAPITAL = 100000          # Starting capital in USD
POSITION_SIZE = 1.0               # BTC per trade
COMMISSION_PERCENT = 0        # Brokerage commission (% per trade)
SLIPPAGE_USD = 0                # Slippage per entry/exit (USD)

# ═════════════════════════════════════════════════════════════════════════════════
# 📅 DATA PARAMETERS
# ═════════════════════════════════════════════════════════════════════════════════

BACKTEST_DAYS = 180               # 6 months
TIMEFRAME = "1min"               # 1-minute candles
SYMBOL = "BTCUSD"                # BTC/USD perpetuals
DATA_CACHE_FILE = 'day_trading_data.pkl'

# ═════════════════════════════════════════════════════════════════════════════════
# 📋 DISPLAY PARAMETERS
# ═════════════════════════════════════════════════════════════════════════════════

# ═════════════════════════════════════════════════════════════════════════════════
# 📋 DISPLAY PARAMETERS
# ═════════════════════════════════════════════════════════════════════════════════

params_display = f"""
🎯 TRADING MODE:
   Mode:                {TRADING_MODE}

⏰ ENTRY & EXIT TIMES (IST):
   Entry Time:          {ENTRY_HOUR:02d}:{ENTRY_MIN:02d}
   Exit Time (EOD):     {EXIT_HOUR:02d}:{EXIT_MIN:02d}
   Expected Trades:     1 per day × 180 days = 180 trades (24/7 market)

📍 POSITION SETTINGS:
   LONG Enabled:        {LONG_ENABLED}
   SHORT Enabled:       {SHORT_ENABLED}

🎯 TAKE PROFIT SETTINGS:
   TP Enabled:          {USE_TAKE_PROFIT}
   TP Mode:             {TP_MODE}
   TP Percent:          {TAKE_PROFIT_PERCENT}%
   TP Points:           ${TAKE_PROFIT_POINTS}

🛑 STOP LOSS SETTINGS:
   SL Enabled:          {USE_STOP_LOSS}
   SL Mode:             {SL_MODE}
   SL Percent:          {STOP_LOSS_PERCENT}%
   SL Points:           ${STOP_LOSS_POINTS}

💰 COSTS & POSITION:
   Initial Capital:     ${INITIAL_CAPITAL:,}
   Position Size:       {POSITION_SIZE} BTC
   Commission:          {COMMISSION_PERCENT}%
   Slippage:            ${SLIPPAGE_USD}

📅 DATA:
   Timeframe:           {TIMEFRAME}
   Symbol:              {SYMBOL}
   Backtest Days:       {BACKTEST_DAYS}
"""

print(params_display)
print("✅ Parameters configured successfully!\n")


                      ⚙️  DAY TRADING STRATEGY PARAMETERS                       


🎯 TRADING MODE:
   Mode:                LONG_ONLY

⏰ ENTRY & EXIT TIMES (IST):
   Entry Time:          10:30
   Exit Time (EOD):     23:58
   Expected Trades:     1 per day × 180 days = 180 trades (24/7 market)

📍 POSITION SETTINGS:
   LONG Enabled:        True
   SHORT Enabled:       False

🎯 TAKE PROFIT SETTINGS:
   TP Enabled:          True
   TP Mode:             POINTS
   TP Percent:          0.4%
   TP Points:           $350

🛑 STOP LOSS SETTINGS:
   SL Enabled:          True
   SL Mode:             POINTS
   SL Percent:          4%
   SL Points:           $10000

💰 COSTS & POSITION:
   Initial Capital:     $100,000
   Position Size:       1.0 BTC
   Commission:          0%
   Slippage:            $0

📅 DATA:
   Timeframe:           1min
   Symbol:              BTCUSD
   Backtest Days:       180

✅ Parameters configured successfully!



---
## BLOCK 3️⃣ : TECHNICAL INDICATORS & STRATEGY LOGIC

In [27]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 3: TECHNICAL INDICATORS & STRATEGY LOGIC
# ═══════════════════════════════════════════════════════════════════════════════

try:
    import talib
    TALIB_AVAILABLE = True
    print("✅ TA-Lib library imported successfully")
except ImportError:
    TALIB_AVAILABLE = False
    print("⚠️  TA-Lib not available, using NumPy alternatives")

print(f"\n" + "="*80)
print("📊 TECHNICAL INDICATORS & DAY TRADING STRATEGY".center(80))
print("="*80 + "\n")

# ═════════════════════════════════════════════════════════════════════════════════
# INDICATOR CALCULATIONS
# ═════════════════════════════════════════════════════════════════════════════════

def calculate_moving_averages(df, periods=[9, 21, 50]):
    """Calculate Simple Moving Averages"""
    df = df.copy()
    for period in periods:
        df[f'SMA_{period}'] = df['close'].rolling(window=period).mean()
    return df

def calculate_rsi(df, period=14):
    """Calculate Relative Strength Index"""
    df = df.copy()
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    return df

def calculate_bollinger_bands(df, period=20):
    """Calculate Bollinger Bands"""
    df = df.copy()
    df['BB_MA'] = df['close'].rolling(window=period).mean()
    df['BB_STD'] = df['close'].rolling(window=period).std()
    df['BB_UPPER'] = df['BB_MA'] + (df['BB_STD'] * 2)
    df['BB_LOWER'] = df['BB_MA'] - (df['BB_STD'] * 2)
    return df

def calculate_atr(df, period=14):
    """Calculate Average True Range"""
    df = df.copy()
    df['TR'] = np.maximum(
        df['high'] - df['low'],
        np.maximum(
            abs(df['high'] - df['close'].shift()),
            abs(df['low'] - df['close'].shift())
        )
    )
    df['ATR'] = df['TR'].rolling(window=period).mean()
    return df

# ═════════════════════════════════════════════════════════════════════════════════
# STRATEGY LOGIC
# ═════════════════════════════════════════════════════════════════════════════════

def is_entry_time(timestamp, entry_time_min):
    """Check if current time matches entry time"""
    ist_time = timestamp + pd.Timedelta(hours=5, minutes=30)
    minutes_in_day = ist_time.hour * 60 + ist_time.minute
    return minutes_in_day == entry_time_min

def is_exit_time(timestamp, exit_time_min):
    """Check if current time is end of day"""
    ist_time = timestamp + pd.Timedelta(hours=5, minutes=30)
    minutes_in_day = ist_time.hour * 60 + ist_time.minute
    return minutes_in_day >= exit_time_min

print("✓ Moving Averages (SMA 9, 21, 50)")
print("✓ RSI (14-period)")
print("✓ Bollinger Bands (20-period)")
print("✓ Average True Range (14-period)")
print("\n✅ Technical indicators ready!\n")

✅ TA-Lib library imported successfully

                 📊 TECHNICAL INDICATORS & DAY TRADING STRATEGY                  

✓ Moving Averages (SMA 9, 21, 50)
✓ RSI (14-period)
✓ Bollinger Bands (20-period)
✓ Average True Range (14-period)

✅ Technical indicators ready!



---
## BLOCK 4️⃣ : DATA FETCHING FROM DELTA EXCHANGE

In [28]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 4: DATA FETCHING FROM DELTA EXCHANGE
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("📥 FETCHING HISTORICAL DATA FROM DELTA EXCHANGE".center(80))
print("="*80 + "\n")

backtest_df = None

# Check cache first
if os.path.exists(DATA_CACHE_FILE):
    print(f"📂 Loading cached data from: {DATA_CACHE_FILE}\n")
    with open(DATA_CACHE_FILE, 'rb') as f:
        backtest_df = pickle.load(f)
    print(f"✓ Data loaded from cache")
    print(f"  Candles: {len(backtest_df):,}")
    print(f"  Date Range: {backtest_df['timestamp'].min()} to {backtest_df['timestamp'].max()}")
    print(f"  Price: ${backtest_df['low'].min():,.2f} - ${backtest_df['high'].max():,.2f}")
    print(f"  Avg Volume: {backtest_df['volume'].mean():,.2f} BTC\n")
else:
    print(f"⏳ Fetching {BACKTEST_DAYS} days of {TIMEFRAME} data from Delta Exchange...\n")
    
    try:
        current_time = int(time.time())
        start_time = current_time - (BACKTEST_DAYS * 24 * 60 * 60)
        
        print(f"📅 Time Range: {datetime.fromtimestamp(start_time, tz=timezone.utc)} to {datetime.fromtimestamp(current_time, tz=timezone.utc)}")
        print(f"📊 Expected Candles: {BACKTEST_DAYS * 24 * 60:,} (1-minute)")
        print(f"🔄 Fetching in batches...\n")
        
        all_candles = []
        end_timestamp = current_time
        request_count = 0
        max_requests = 200
        
        while end_timestamp > start_time and request_count < max_requests:
            request_count += 1
            batch_start = max(end_timestamp - (2000 * 60), start_time)
            
            params = {
                'resolution': '1m',
                'symbol': SYMBOL,
                'start': batch_start,
                'end': end_timestamp
            }
            
            try:
                response = requests.get(HISTORY_URL, params=params, timeout=15)
                if response.status_code == 200:
                    data = response.json()
                    if data.get('success') and data.get('result'):
                        candles = data['result']
                        all_candles.extend(candles)
                        oldest_time = candles[-1]['time']
                        progress = min(100, ((current_time - oldest_time) / (current_time - start_time)) * 100)
                        
                        if request_count % 20 == 0 or request_count == 1:
                            print(f"  Request {request_count}: {len(candles)} candles | Total: {len(all_candles):,} | {progress:.1f}%")
                        
                        end_timestamp = oldest_time - 60
                        time.sleep(0.3)
                    else:
                        break
                else:
                    break
            except Exception as e:
                print(f"  ⚠️  Request {request_count} failed: {str(e)}")
                break
        
        print(f"\n✓ Fetch Complete: {request_count} API requests | {len(all_candles):,} candles\n")
        
        if len(all_candles) > 100:
            backtest_df = pd.DataFrame(all_candles)
            backtest_df['timestamp'] = pd.to_datetime(backtest_df['time'], unit='s', utc=True)
            backtest_df = backtest_df[['timestamp', 'open', 'high', 'low', 'close', 'volume']].copy()
            backtest_df = backtest_df.sort_values('timestamp').reset_index(drop=True)
            
            # Save to cache
            with open(DATA_CACHE_FILE, 'wb') as f:
                pickle.dump(backtest_df, f)
            print(f"💾 Data cached to: {DATA_CACHE_FILE}\n")
        else:
            print("⚠️  Insufficient data fetched")
            backtest_df = None
    
    except Exception as e:
        print(f"✗ Error fetching data: {e}")
        backtest_df = None

if backtest_df is not None:
    print("╔" + "═"*78 + "╗")
    print("║" + "✅ DATA READY FOR BACKTESTING".center(78) + "║")
    print("╚" + "═"*78 + "╝")
    print(f"\n📊 Data Summary:")
    print(f"   Total Candles:     {len(backtest_df):,}")
    print(f"   Time Range:        {backtest_df['timestamp'].min()} to {backtest_df['timestamp'].max()}")
    print(f"   Duration:          {(backtest_df['timestamp'].max() - backtest_df['timestamp'].min()).days} days")
    print(f"   Price Range:       ${backtest_df['low'].min():,.2f} - ${backtest_df['high'].max():,.2f}")
    print(f"   Avg Daily Volume:  {backtest_df['volume'].mean():,.2f} BTC")
    print(f"   Total Volume:      {backtest_df['volume'].sum():,.2f} BTC\n")
else:
    print("❌ Failed to load data. Check API credentials and try Block 4 again.")


                 📥 FETCHING HISTORICAL DATA FROM DELTA EXCHANGE                 

📂 Loading cached data from: day_trading_data.pkl

✓ Data loaded from cache
  Candles: 259,199
  Date Range: 2025-09-27 12:54:00+00:00 to 2026-03-26 12:52:00+00:00
  Price: $59,835.50 - $126,205.50
  Avg Volume: 7,870.53 BTC

╔══════════════════════════════════════════════════════════════════════════════╗
║                         ✅ DATA READY FOR BACKTESTING                         ║
╚══════════════════════════════════════════════════════════════════════════════╝

📊 Data Summary:
   Total Candles:     259,199
   Time Range:        2025-09-27 12:54:00+00:00 to 2026-03-26 12:52:00+00:00
   Duration:          179 days
   Price Range:       $59,835.50 - $126,205.50
   Avg Daily Volume:  7,870.53 BTC
   Total Volume:      2,040,032,518.00 BTC



---
## BLOCK 5️⃣ : BACKTEST ENGINE WITH BACKTRADER

In [29]:

# ═══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC BLOCK: ANALYZE DATA COVERAGE & UNIQUE DAYS
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("🔍 DATA COVERAGE ANALYSIS - CHECKING FOR GAPS & UNIQUE DAYS".center(80))
print("="*80 + "\n")

if backtest_df is not None:
    # Convert to IST for analysis
    backtest_df_analysis = backtest_df.copy()
    backtest_df_analysis['timestamp_ist'] = backtest_df_analysis['timestamp'] + pd.Timedelta(hours=5, minutes=30)
    backtest_df_analysis['date_ist'] = backtest_df_analysis['timestamp_ist'].dt.date
    backtest_df_analysis['hour_ist'] = backtest_df_analysis['timestamp_ist'].dt.hour
    backtest_df_analysis['minute_ist'] = backtest_df_analysis['timestamp_ist'].dt.minute
    
    # Get unique dates
    unique_dates = backtest_df_analysis['date_ist'].nunique()
    date_range_start = backtest_df_analysis['date_ist'].min()
    date_range_end = backtest_df_analysis['date_ist'].max()
    total_days_in_range = (date_range_end - date_range_start).days + 1
    
    print(f"📅 DATA COVERAGE REPORT:")
    print(f"   Start Date (IST):         {date_range_start}")
    print(f"   End Date (IST):           {date_range_end}")
    print(f"   Calendar Days in Range:   {total_days_in_range}")
    print(f"   Unique Days with Data:    {unique_dates}")
    print(f"   Missing Days:             {total_days_in_range - unique_dates}")
    print()
    
    # Check if we have data for the entry hour
    entry_hour_data = backtest_df_analysis[backtest_df_analysis['hour_ist'] == ENTRY_HOUR]
    unique_entry_hour_dates = entry_hour_data['date_ist'].nunique()
    
    print(f"📍 ENTRY HOUR ({ENTRY_HOUR}:00 IST) COVERAGE:")
    print(f"   Candles at entry hour:    {len(entry_hour_data):,}")
    print(f"   Unique dates with entry:  {unique_entry_hour_dates}")
    print(f"   % of days with entry:     {(unique_entry_hour_dates / unique_dates * 100):.1f}%")
    print()
    
    # Check for missing hours (data gaps)
    all_hours = backtest_df_analysis['hour_ist'].unique()
    all_hours_expected = set(range(24))
    missing_hours = sorted(all_hours_expected - set(all_hours))
    
    if missing_hours:
        print(f"⚠️  ALERT: Missing data for hours: {missing_hours}")
    else:
        print(f"✅ All 24 hours covered in the dataset")
    print()
    
    # Detailed breakdown by entry hour
    print(f"📊 DETAILED BREAKDOWN FOR ENTRY HOUR {ENTRY_HOUR}:00 IST:")
    entry_minute_range = backtest_df_analysis[
        (backtest_df_analysis['hour_ist'] == ENTRY_HOUR) &
        (backtest_df_analysis['minute_ist'] == ENTRY_MIN)
    ]
    print(f"   Exact entry time candles: {len(entry_minute_range):,}")
    
    if len(entry_minute_range) > 0:
        entry_dates = entry_minute_range['date_ist'].value_counts().sort_index()
        print(f"   Dates covered:            {len(entry_dates)}")
        print(f"   Avg candles per date:     {len(entry_minute_range) / len(entry_dates):.1f}")
    
    print("\n" + "="*80)
    print(f"💡 EXPECTED vs ACTUAL:")
    print(f"   Requested Backtest Days:  {BACKTEST_DAYS}")
    print(f"   Actual Unique Days:       {unique_dates}")
    print(f"   Expected Trades (1/day):  {BACKTEST_DAYS}")
    print(f"   Trades to Generate:       {unique_entry_hour_dates}")
    print("="*80 + "\n")

else:
    print("❌ No data available for analysis")




           🔍 DATA COVERAGE ANALYSIS - CHECKING FOR GAPS & UNIQUE DAYS           

📅 DATA COVERAGE REPORT:
   Start Date (IST):         2025-09-27
   End Date (IST):           2026-03-26
   Calendar Days in Range:   181
   Unique Days with Data:    181
   Missing Days:             0

📍 ENTRY HOUR (10:00 IST) COVERAGE:
   Candles at entry hour:    10,800
   Unique dates with entry:  180
   % of days with entry:     99.4%

✅ All 24 hours covered in the dataset

📊 DETAILED BREAKDOWN FOR ENTRY HOUR 10:00 IST:
   Exact entry time candles: 180
   Dates covered:            180
   Avg candles per date:     1.0

💡 EXPECTED vs ACTUAL:
   Requested Backtest Days:  180
   Actual Unique Days:       181
   Expected Trades (1/day):  180
   Trades to Generate:       180



In [30]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 5: BACKTEST ENGINE - DAY TRADING STRATEGY
# ═══════════════════════════════════════════════════════════════════════════════

import backtrader as bt

print("\n" + "="*80)
print("🚀 RUNNING BACKTEST - DAY TRADING STRATEGY".center(80))
print("="*80 + "\n")

# ═════════════════════════════════════════════════════════════════════════════════
# TRADE TRACKING DATA STRUCTURE
# ═════════════════════════════════════════════════════════════════════════════════

all_trades = []

# ═════════════════════════════════════════════════════════════════════════════════
# DAY TRADING STRATEGY
# ═════════════════════════════════════════════════════════════════════════════════

class DayTradingStrategy(bt.Strategy):
    """
    Day Trading Strategy - Entry at midnight, Exit at EOD
    TRACKS ALL TRADES FOR LOGS AND VISUALIZATION
    Supports both PERCENTAGE and POINTS based TP/SL
    """
    
    params = (
        ('entry_time', ENTRY_TIME_IN_MIN),
        ('exit_time', EXIT_TIME_IN_MIN),
        ('long_enabled', LONG_ENABLED),
        ('short_enabled', SHORT_ENABLED),
        ('use_tp', USE_TAKE_PROFIT),
        ('tp_mode', TP_MODE),
        ('tp_percent', TAKE_PROFIT_PERCENT),
        ('tp_points', TAKE_PROFIT_POINTS),
        ('use_sl', USE_STOP_LOSS),
        ('sl_mode', SL_MODE),
        ('sl_percent', STOP_LOSS_PERCENT),
        ('sl_points', STOP_LOSS_POINTS),
    )
    
    def __init__(self):
        # Minimal indicators - use short period to avoid warmup delays
        # These are for logging only, not used in entry logic
        self.sma9 = bt.indicators.SimpleMovingAverage(self.data.close, period=1)  # No warmup needed
        self.sma21 = bt.indicators.SimpleMovingAverage(self.data.close, period=1)
        self.sma50 = bt.indicators.SimpleMovingAverage(self.data.close, period=1)
        
        self.entry_price = None
        self.entry_time = None
        self.entry_type = None
        self.trades_count = 0
        self.last_entry_day = -1  # Track last entry day
        self.max_price_since_entry = None
        self.min_price_since_entry = None
    
    def get_ist_time(self):
        """Get current time in IST"""
        current_time = self.data.datetime.datetime(0)
        ist_time = current_time + timedelta(hours=5, minutes=30)
        return ist_time
    
    def get_minutes_in_day(self, timestamp):
        """Get minutes from midnight in IST"""
        return timestamp.hour * 60 + timestamp.minute
    
    def next(self):
        current_price = self.data.close[0]
        current_time = self.get_ist_time()
        minutes_in_day = self.get_minutes_in_day(current_time)
        current_day = current_time.date()
        
        # ═════════════════════════════════════════════════════════════════════════════
        # EXIT LOGIC
        # ═════════════════════════════════════════════════════════════════════════════
        
        if self.position:
            # Update favorable/adverse prices
            if self.position.size > 0:  # LONG
                if self.max_price_since_entry is None:
                    self.max_price_since_entry = current_price
                else:
                    self.max_price_since_entry = max(self.max_price_since_entry, current_price)
                self.min_price_since_entry = current_price
            else:  # SHORT
                if self.min_price_since_entry is None:
                    self.min_price_since_entry = current_price
                else:
                    self.min_price_since_entry = min(self.min_price_since_entry, current_price)
                self.max_price_since_entry = current_price
            
            exit_signal = False
            exit_reason = ""
            
            # LONG EXIT
            if self.position.size > 0:
                # Take Profit (Percentage)
                if self.p.use_tp and self.p.tp_mode in ["PERCENTAGE", "BOTH"] and self.entry_price:
                    tp_price_pct = self.entry_price * (1 + self.p.tp_percent / 100)
                    if current_price >= tp_price_pct:
                        exit_signal = True
                        exit_reason = "Take Profit (Percent)"
                
                # Take Profit (Points)
                if self.p.use_tp and self.p.tp_mode in ["POINTS", "BOTH"] and self.entry_price and not exit_signal:
                    tp_price_points = self.entry_price + (self.p.tp_points / POSITION_SIZE)
                    if current_price >= tp_price_points:
                        exit_signal = True
                        exit_reason = "Take Profit (Points)"
                
                # Stop Loss (Percentage)
                if self.p.use_sl and self.p.sl_mode in ["PERCENTAGE", "BOTH"] and self.entry_price and not exit_signal:
                    sl_price_pct = self.entry_price * (1 - self.p.sl_percent / 100)
                    if current_price <= sl_price_pct:
                        exit_signal = True
                        exit_reason = "Stop Loss (Percent)"
                
                # Stop Loss (Points)
                if self.p.use_sl and self.p.sl_mode in ["POINTS", "BOTH"] and self.entry_price and not exit_signal:
                    sl_price_points = self.entry_price - (self.p.sl_points / POSITION_SIZE)
                    if current_price <= sl_price_points:
                        exit_signal = True
                        exit_reason = "Stop Loss (Points)"
                
                # End of Day Exit
                if not exit_signal and minutes_in_day >= self.p.exit_time:
                    exit_signal = True
                    exit_reason = "End of Day"
            
            # SHORT EXIT
            elif self.position.size < 0:
                # Take Profit (Percentage)
                if self.p.use_tp and self.p.tp_mode in ["PERCENTAGE", "BOTH"] and self.entry_price:
                    tp_price_pct = self.entry_price * (1 - self.p.tp_percent / 100)
                    if current_price <= tp_price_pct:
                        exit_signal = True
                        exit_reason = "Take Profit (Percent)"
                
                # Take Profit (Points)
                if self.p.use_tp and self.p.tp_mode in ["POINTS", "BOTH"] and self.entry_price and not exit_signal:
                    tp_price_points = self.entry_price - (self.p.tp_points / POSITION_SIZE)
                    if current_price <= tp_price_points:
                        exit_signal = True
                        exit_reason = "Take Profit (Points)"
                
                # Stop Loss (Percentage)
                if self.p.use_sl and self.p.sl_mode in ["PERCENTAGE", "BOTH"] and self.entry_price and not exit_signal:
                    sl_price_pct = self.entry_price * (1 + self.p.sl_percent / 100)
                    if current_price >= sl_price_pct:
                        exit_signal = True
                        exit_reason = "Stop Loss (Percent)"
                
                # Stop Loss (Points)
                if self.p.use_sl and self.p.sl_mode in ["POINTS", "BOTH"] and self.entry_price and not exit_signal:
                    sl_price_points = self.entry_price + (self.p.sl_points / POSITION_SIZE)
                    if current_price >= sl_price_points:
                        exit_signal = True
                        exit_reason = "Stop Loss (Points)"
                
                # End of Day Exit
                if not exit_signal and minutes_in_day >= self.p.exit_time:
                    exit_signal = True
                    exit_reason = "End of Day"
            
            if exit_signal:
                # Calculate trade metrics
                if self.entry_type == 'LONG':
                    pnl = (current_price - self.entry_price - SLIPPAGE_USD) * POSITION_SIZE
                    pnl_pct = (pnl / (self.entry_price * POSITION_SIZE)) * 100
                    favorable = (self.max_price_since_entry - self.entry_price) * POSITION_SIZE
                    adverse = (self.entry_price - self.min_price_since_entry) * POSITION_SIZE
                else:  # SHORT
                    pnl = (self.entry_price - current_price - SLIPPAGE_USD) * POSITION_SIZE
                    pnl_pct = (pnl / (self.entry_price * POSITION_SIZE)) * 100
                    favorable = (self.entry_price - self.min_price_since_entry) * POSITION_SIZE
                    adverse = (self.max_price_since_entry - self.entry_price) * POSITION_SIZE
                
                # Store trade details
                trade_record = {
                    'trade_num': self.trades_count,
                    'entry_time': self.entry_time,
                    'entry_price': self.entry_price,
                    'entry_type': self.entry_type,
                    'exit_time': current_time,
                    'exit_price': current_price,
                    'exit_reason': exit_reason,
                    'position_size': POSITION_SIZE,
                    'pnl': pnl,
                    'pnl_pct': pnl_pct,
                    'favorable': favorable,
                    'adverse': adverse,
                    'max_price': self.max_price_since_entry,
                    'min_price': self.min_price_since_entry,
                }
                
                all_trades.append(trade_record)
                self.close()
                self.max_price_since_entry = None
                self.min_price_since_entry = None
                return
        
        # ═════════════════════════════════════════════════════════════════════════════
        # ENTRY LOGIC - One trade per calendar day at entry time
        # ═════════════════════════════════════════════════════════════════════════════
        
        # Entry: Only at specified time, once per day
        if not self.position and minutes_in_day == self.p.entry_time:
            # Check if we already traded today (for 24/7 market)
            if self.last_entry_day != current_day.toordinal():
                # LONG ENTRY
                if self.p.long_enabled:
                    self.buy(size=POSITION_SIZE)
                    self.entry_type = 'LONG'
                    self.entry_price = current_price
                    self.entry_time = current_time
                    self.trades_count += 1
                    self.last_entry_day = current_day.toordinal()
                    self.max_price_since_entry = current_price
                    self.min_price_since_entry = current_price
                # SHORT ENTRY
                elif self.p.short_enabled:
                    self.sell(size=POSITION_SIZE)
                    self.entry_type = 'SHORT'
                    self.entry_price = current_price
                    self.entry_time = current_time
                    self.trades_count += 1
                    self.last_entry_day = current_day.toordinal()
                    self.max_price_since_entry = current_price
                    self.min_price_since_entry = current_price

# ═════════════════════════════════════════════════════════════════════════════════
# RUN BACKTEST
# ═════════════════════════════════════════════════════════════════════════════════

if backtest_df is not None:
    all_trades = []
    
    print("📋 STRATEGY CONFIGURATION (from Block 2):")
    print(f"   Trading Mode:            {TRADING_MODE}")
    print(f"   Entry Time (IST):        {ENTRY_HOUR:02d}:{ENTRY_MIN:02d}")
    print(f"   Exit Time (IST):         {EXIT_HOUR:02d}:{EXIT_MIN:02d}")
    print(f"   TP Mode:                 {TP_MODE} (Percent: {TAKE_PROFIT_PERCENT}%, Points: ${TAKE_PROFIT_POINTS})")
    print(f"   SL Mode:                 {SL_MODE} (Percent: {STOP_LOSS_PERCENT}%, Points: ${STOP_LOSS_POINTS})")
    print(f"   Position Size:           {POSITION_SIZE} BTC")
    print(f"   Commission:              {COMMISSION_PERCENT}%")
    print(f"   Slippage:                ${SLIPPAGE_USD}")
    print(f"   Capital:                 ${INITIAL_CAPITAL:,}\n")
    
    # Prepare data for Backtrader
    df_bt = backtest_df.set_index('timestamp')
    df_bt.index.name = 'datetime'
    
    # Create Cerebro engine
    cerebro = bt.Cerebro()
    data_feed = bt.feeds.PandasData(dataname=df_bt)
    cerebro.adddata(data_feed)
    
    # Add strategy - Disable default minimum bars to allow trading from day 1
    cerebro.addstrategy(DayTradingStrategy)
    cerebro.broker.setcash(INITIAL_CAPITAL)
    cerebro.broker.setcommission(COMMISSION_PERCENT / 100)
    
    # Add analyzers
    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
    cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
    
    print("⏳ Running Day Trading backtest...\n")
    results = cerebro.run()
    strat = results[0]
    
    final_value = cerebro.broker.getvalue()
    net_profit = final_value - INITIAL_CAPITAL
    roi = (net_profit / INITIAL_CAPITAL) * 100
    
    # Extract analytics
    trade_analysis = strat.analyzers.trades.get_analysis()
    total_trades = trade_analysis.get('total', {}).get('total', 0)
    win_trades = trade_analysis.get('won', {}).get('total', 0)
    loss_trades = trade_analysis.get('lost', {}).get('total', 0)
    
    max_dd_data = strat.analyzers.drawdown.get_analysis()
    max_dd = max_dd_data.get('max', {}).get('drawdown', 0)
    
    win_rate = (win_trades / total_trades * 100) if total_trades > 0 else 0
    
    # Profit factor
    if trade_analysis.get('lost') and trade_analysis.get('lost').get('pnl'):
        losing_pnl = abs(trade_analysis['lost']['pnl']['total'])
        winning_pnl = trade_analysis.get('won', {}).get('pnl', {}).get('total', 0)
        profit_factor = winning_pnl / losing_pnl if losing_pnl > 0 else 0
    else:
        profit_factor = 0
    
    print("╔" + "═"*78 + "╗")
    print("║" + "✅ DAY TRADING BACKTEST RESULTS".center(78) + "║")
    print("╚" + "═"*78 + "╝\n")
    
    print(f"💰 CAPITAL METRICS:")
    print(f"   Starting Capital:      ${INITIAL_CAPITAL:>12,.2f}")
    print(f"   Final Capital:         ${final_value:>12,.2f}")
    print(f"   Net Profit/Loss:       ${net_profit:>12,.2f}")
    print(f"   ROI:                   {roi:>13.2f}%\n")
    
    print(f"📈 TRADE METRICS:")
    print(f"   Total Trades:          {len(all_trades):>15}")
    print(f"   Winning Trades:        {win_trades:>15}")
    print(f"   Losing Trades:         {loss_trades:>15}")
    print(f"   Win Rate:              {win_rate:>14.2f}%")
    print(f"   Profit Factor:         {profit_factor:>14.2f}\n")
    
    print(f"📉 RISK METRICS:")
    print(f"   Max Drawdown:          {max_dd:>14.2f}%\n")
    
    print(f"✅ Backtest completed successfully!")
    print(f"📊 {len(all_trades)} actual trades recorded for visualization")
    print(f"💡 Try modifying Block 2 parameters and re-run this block to see new results!\n")
else:
    print("❌ No data available for backtest")


                   🚀 RUNNING BACKTEST - DAY TRADING STRATEGY                    

📋 STRATEGY CONFIGURATION (from Block 2):
   Trading Mode:            LONG_ONLY
   Entry Time (IST):        10:30
   Exit Time (IST):         23:58
   TP Mode:                 POINTS (Percent: 0.4%, Points: $350)
   SL Mode:                 POINTS (Percent: 4%, Points: $10000)
   Position Size:           1.0 BTC
   Commission:              0%
   Slippage:                $0
   Capital:                 $100,000

⏳ Running Day Trading backtest...

╔══════════════════════════════════════════════════════════════════════════════╗
║                        ✅ DAY TRADING BACKTEST RESULTS                        ║
╚══════════════════════════════════════════════════════════════════════════════╝

💰 CAPITAL METRICS:
   Starting Capital:      $  100,000.00
   Final Capital:         $   78,951.50
   Net Profit/Loss:       $  -21,048.50
   ROI:                          -21.05%

📈 TRADE METRICS:
   Total Trades:           

---
## BLOCK 6️⃣ : TRADINGVIEW-STYLE TRADE LOGS WITH EXIT REASONS

In [31]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 6: TRADINGVIEW-STYLE TRADE LOGS WITH COMPREHENSIVE ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*140)
print("📊 PROFESSIONAL TRADE ANALYSIS - TRADINGVIEW STYLE".center(140))
print("="*140 + "\n")

# ═════════════════════════════════════════════════════════════════════════════════
# CREATE COMPREHENSIVE TRADE LOG FROM ACTUAL BACKTEST TRADES
# ═════════════════════════════════════════════════════════════════════════════════

def generate_comprehensive_trade_log(trades_list):
    """Generate comprehensive trade log with TradingView-style metrics"""
    
    if not trades_list or len(trades_list) == 0:
        print("⚠️  NO TRADES EXECUTED IN THIS BACKTEST")
        print("   ────────────────────────────────────────────────")
        print("   Try adjusting parameters in Block 2:")
        print(f"   • Entry Time: {ENTRY_HOUR:02d}:{ENTRY_MIN:02d} IST")
        print(f"   • Exit Time: {EXIT_HOUR:02d}:{EXIT_MIN:02d} IST")
        print(f"   • TP: {USE_TAKE_PROFIT} ({TP_MODE})")
        print(f"   • SL: {USE_STOP_LOSS} ({SL_MODE})\n")
        return None
    
    trades = []
    cumulative_pnl = 0
    consecutive_wins = 0
    consecutive_losses = 0
    max_consecutive_wins = 0
    max_consecutive_losses = 0
    
    for i, trade in enumerate(trades_list):
        cumulative_pnl += trade['pnl']
        
        # Track consecutive wins/losses
        if trade['pnl'] > 0:
            consecutive_wins += 1
            consecutive_losses = 0
            max_consecutive_wins = max(max_consecutive_wins, consecutive_wins)
        else:
            consecutive_losses += 1
            consecutive_wins = 0
            max_consecutive_losses = max(max_consecutive_losses, consecutive_losses)
        
        trades.append({
            'Trade #': i + 1,
            'Type': trade['entry_type'],
            'Entry Time': trade['entry_time'].strftime('%b %d, %Y %H:%M'),
            'Entry Price': trade['entry_price'],
            'Exit Time': trade['exit_time'].strftime('%b %d, %Y %H:%M'),
            'Exit Price': trade['exit_price'],
            'Exit Reason': trade['exit_reason'],
            'Position Size': trade['position_size'],
            'Net P&L (USD)': trade['pnl'],
            'Net P&L (%)': trade['pnl_pct'],
            'Favorable': trade['favorable'],
            'Adverse': trade['adverse'],
            'R:R Ratio': trade['favorable'] / trade['adverse'] if trade['adverse'] > 0 else 0,
            'Cumulative P&L': cumulative_pnl,
            'Cumulative P&L (%)': (cumulative_pnl / INITIAL_CAPITAL) * 100,
            'Max Price': trade['max_price'],
            'Min Price': trade['min_price'],
            'Price Range': trade['max_price'] - trade['min_price'],
        })
    
    df = pd.DataFrame(trades)
    return df, max_consecutive_wins, max_consecutive_losses

if backtest_df is not None and len(all_trades) > 0:
    result = generate_comprehensive_trade_log(all_trades)
    
    if result is not None:
        trade_log, max_cons_wins, max_cons_losses = result
        
        # ═════════════════════════════════════════════════════════════════════════════
        # 📊 SECTION 1: PERFORMANCE SUMMARY
        # ═════════════════════════════════════════════════════════════════════════════
        
        print("╔" + "═"*138 + "╗")
        print("║" + "📊 PERFORMANCE SUMMARY".ljust(139) + "║")
        print("╚" + "═"*138 + "╝\n")
        
        total_pnl = trade_log['Net P&L (USD)'].sum()
        total_pnl_pct = (total_pnl / INITIAL_CAPITAL) * 100
        final_capital = INITIAL_CAPITAL + total_pnl
        
        winning_trades = (trade_log['Net P&L (USD)'] > 0).sum()
        losing_trades = (trade_log['Net P&L (USD)'] < 0).sum()
        breakeven_trades = (trade_log['Net P&L (USD)'] == 0).sum()
        
        win_rate = (winning_trades / len(trade_log) * 100) if len(trade_log) > 0 else 0
        
        avg_win = trade_log[trade_log['Net P&L (USD)'] > 0]['Net P&L (USD)'].mean() if winning_trades > 0 else 0
        avg_loss = trade_log[trade_log['Net P&L (USD)'] < 0]['Net P&L (USD)'].mean() if losing_trades > 0 else 0
        largest_win = trade_log['Net P&L (USD)'].max()
        largest_loss = trade_log['Net P&L (USD)'].min()
        
        profit_factor = abs(trade_log[trade_log['Net P&L (USD)'] > 0]['Net P&L (USD)'].sum() / 
                           trade_log[trade_log['Net P&L (USD)'] < 0]['Net P&L (USD)'].sum()) \
                       if losing_trades > 0 else 0
        
        expectancy = total_pnl / len(trade_log) if len(trade_log) > 0 else 0
        
        max_drawdown = trade_log['Cumulative P&L (%)'].min()
        
        # Display metrics in organized columns
        col1 = f"""
🎯 CAPITAL & RETURNS:
   Starting Capital:    ${INITIAL_CAPITAL:>12,.2f}
   Final Capital:       ${final_capital:>12,.2f}
   Net Profit/Loss:     ${total_pnl:>12,.2f}
   Return on Capital:   {total_pnl_pct:>13.2f}%
   
📈 WIN METRICS:
   Total Trades:        {len(trade_log):>15}
   Winning Trades:      {winning_trades:>15} ✓
   Losing Trades:       {losing_trades:>15} ✗
   Breakeven Trades:    {breakeven_trades:>15}
   Win Rate:            {win_rate:>14.2f}%
"""
        
        col2 = f"""
💰 PROFIT METRICS:
   Largest Win:         ${largest_win:>12,.2f}
   Average Win:         ${avg_win:>12,.2f}
   Largest Loss:        ${largest_loss:>12,.2f}
   Average Loss:        ${avg_loss:>12,.2f}
   Profit Factor:       {profit_factor:>14.2f}
   
📉 RISK METRICS:
   Expectancy/Trade:    ${expectancy:>12,.2f}
   Max Drawdown:        {max_drawdown:>14.2f}%
   Max Cons. Wins:      {max_cons_wins:>15}
   Max Cons. Losses:    {max_cons_losses:>15}
"""
        
        # Print side by side
        col1_lines = col1.strip().split('\n')
        col2_lines = col2.strip().split('\n')
        max_lines = max(len(col1_lines), len(col2_lines))
        
        for i in range(max_lines):
            line1 = col1_lines[i] if i < len(col1_lines) else ""
            line2 = col2_lines[i] if i < len(col2_lines) else ""
            print(f"{line1:<70} {line2}")
        
        print()
        
        # ═════════════════════════════════════════════════════════════════════════════
        # 📋 SECTION 2: EXIT REASON ANALYSIS
        # ═════════════════════════════════════════════════════════════════════════════
        
        print("\n" + "╔" + "═"*138 + "╗")
        print("║" + "📋 EXIT REASON BREAKDOWN".ljust(139) + "║")
        print("╚" + "═"*138 + "╝\n")
        
        exit_reasons = trade_log['Exit Reason'].value_counts()
        tp_count = (trade_log['Exit Reason'].str.contains('Profit')).sum()
        sl_count = (trade_log['Exit Reason'].str.contains('Stop')).sum()
        eod_count = (trade_log['Exit Reason'].str.contains('End of Day')).sum()
        
        # Calculate P&L by exit reason
        pnl_by_reason = {}
        for reason in exit_reasons.index:
            pnl_by_reason[reason] = trade_log[trade_log['Exit Reason'] == reason]['Net P&L (USD)'].sum()
        
        print(f"{'Exit Reason':<30} {'Count':>10} {'% of Trades':>15} {'Total P&L':>15} {'Avg P&L/Trade':>18}")
        print("-" * 88)
        
        for reason in exit_reasons.index:
            count = exit_reasons[reason]
            pct = (count / len(trade_log)) * 100
            total_pnl_reason = pnl_by_reason[reason]
            avg_pnl_reason = total_pnl_reason / count if count > 0 else 0
            
            print(f"{reason:<30} {count:>10} {pct:>14.1f}% ${total_pnl_reason:>13,.2f} ${avg_pnl_reason:>16,.2f}")
        
        print()
        
        # ═════════════════════════════════════════════════════════════════════════════
        # 🎯 SECTION 3: TRADE TYPE ANALYSIS (LONG vs SHORT)
        # ═════════════════════════════════════════════════════════════════════════════
        
        print("\n" + "╔" + "═"*138 + "╗")
        print("║" + "🎯 TRADE TYPE ANALYSIS".ljust(139) + "║")
        print("╚" + "═"*138 + "╝\n")
        
        long_trades = trade_log[trade_log['Type'] == 'LONG']
        short_trades = trade_log[trade_log['Type'] == 'SHORT']
        
        print(f"{'Trade Type':<20} {'Count':>10} {'Win %':>10} {'Avg P&L':>15} {'Total P&L':>15} {'ROI %':>10}")
        print("-" * 80)
        
        for trade_type, trades_subset in [('LONG', long_trades), ('SHORT', short_trades)]:
            if len(trades_subset) > 0:
                count = len(trades_subset)
                win_pct = (trades_subset['Net P&L (USD)'] > 0).sum() / count * 100
                avg_pnl = trades_subset['Net P&L (USD)'].mean()
                total_pnl_type = trades_subset['Net P&L (USD)'].sum()
                roi_pct = (total_pnl_type / INITIAL_CAPITAL) * 100
                
                print(f"{trade_type:<20} {count:>10} {win_pct:>9.1f}% ${avg_pnl:>13,.2f} ${total_pnl_type:>13,.2f} {roi_pct:>9.2f}%")
        
        print()
        
        # ═════════════════════════════════════════════════════════════════════════════
        # ⏰ SECTION 4: TRADE DURATION & MOVEMENT
        # ═════════════════════════════════════════════════════════════════════════════
        
        print("\n" + "╔" + "═"*138 + "╗")
        print("║" + "⏰ TRADE DURATION & PRICE MOVEMENT".ljust(139) + "║")
        print("╚" + "═"*138 + "╝\n")
        
        trade_log['Duration (min)'] = (pd.to_datetime(trade_log['Exit Time']) - 
                                       pd.to_datetime(trade_log['Entry Time'])).dt.total_seconds() / 60
        
        avg_duration = trade_log['Duration (min)'].mean()
        avg_favorable = trade_log['Favorable'].mean()
        avg_adverse = trade_log['Adverse'].mean()
        avg_rr_ratio = trade_log['R:R Ratio'].mean()
        
        print(f"Average Trade Duration:      {avg_duration:>10.0f} minutes")
        print(f"Average Favorable Move:      ${avg_favorable:>15,.2f}")
        print(f"Average Adverse Move:        ${avg_adverse:>15,.2f}")
        print(f"Average R:R Ratio:           {avg_rr_ratio:>16.2f}")
        print()
        
        # ═════════════════════════════════════════════════════════════════════════════
        # 📊 SECTION 5: DETAILED TRADE TABLE
        # ═════════════════════════════════════════════════════════════════════════════
        
        print("\n" + "╔" + "═"*138 + "╗")
        print("║" + "📊 DETAILED TRADE LOG (MOST RECENT FIRST)".ljust(139) + "║")
        print("╚" + "═"*138 + "╝\n")
        
        # Create display version
        for idx, (i, trade) in enumerate(trade_log.iloc[::-1].iterrows()):
            status = "✓ WIN" if trade['Net P&L (USD)'] > 0 else ("✗ LOSS" if trade['Net P&L (USD)'] < 0 else "= B/E")
            status_color = "✓" if trade['Net P&L (USD)'] > 0 else "✗"
            
            print(f"\n┌─ Trade #{int(trade['Trade #']):3d} | {status:^12} ┐")
            print(f"│ Type: {trade['Type']:<8} │ Entry Time: {trade['Entry Time']:<20} │ Exit Time: {trade['Exit Time']:<20} │")
            print(f"│ Entry: ${trade['Entry Price']:>10,.2f} │ Exit: ${trade['Exit Price']:>10,.2f} │ Position: {trade['Position Size']:>5.2f} BTC")
            print(f"│")
            print(f"│ P&L: ${trade['Net P&L (USD)']:>10,.2f} ({trade['Net P&L (%)']:>7.2f}%) │ Exit Reason: {trade['Exit Reason']:<20} │")
            print(f"│ Favorable: ${trade['Favorable']:>10,.2f} │ Adverse: ${trade['Adverse']:>10,.2f} │ Risk:Reward: {trade['R:R Ratio']:>6.2f}")
            print(f"│ Cumulative P&L: ${trade['Cumulative P&L']:>10,.2f} ({trade['Cumulative P&L (%)']:>7.2f}%)")
            print(f"└────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘")
            
            if idx >= 19:  # Show only last 20 trades
                remaining = len(trade_log) - idx - 1
                if remaining > 0:
                    print(f"\n... and {remaining} more trades (scroll up to see all)\n")
                break
        
        print("\n" + "="*140)
        
        # Export to CSV
        csv_file = 'day_trading_log.csv'
        export_df = trade_log[[
            'Trade #', 'Type', 'Entry Time', 'Entry Price', 'Exit Time', 'Exit Price', 
            'Exit Reason', 'Position Size', 'Net P&L (USD)', 'Net P&L (%)', 
            'Favorable', 'Adverse', 'R:R Ratio', 'Cumulative P&L', 'Cumulative P&L (%)'
        ]]
        export_df.to_csv(csv_file, index=False)
        
        print(f"\n✅ Trade log exported to: {csv_file}")
        print(f"📈 Total records: {len(trade_log)} trades\n")
        
        # Store for use in Block 7
        backtest_trade_log = trade_log
    else:
        print("❌ Could not generate trade log")
        backtest_trade_log = None
else:
    if backtest_df is not None:
        print("⚠️  NO TRADES WERE EXECUTED IN THE BACKTEST")
        print("\n" + "═"*140)
        print("POSSIBLE REASONS:")
        print("═"*140)
        print(f"""
1. ENTRY TIME NOT REACHED
   • Entry Time: {ENTRY_HOUR:02d}:{ENTRY_MIN:02d} IST
   • Check if this time exists in your data backtest period
   
2. TRADING MODE CONFLICT
   • LONG Enabled: {LONG_ENABLED}
   • SHORT Enabled: {SHORT_ENABLED}
   • At least one must be True
   
3. TAKE PROFIT / STOP LOSS TOO RESTRICTIVE
   • TP Mode: {TP_MODE} (Percent: {TAKE_PROFIT_PERCENT}%, Points: ${TAKE_PROFIT_POINTS})
   • SL Mode: {SL_MODE} (Percent: {STOP_LOSS_PERCENT}%, Points: ${STOP_LOSS_POINTS})
   • May exit immediately if conditions are too easy
   
4. DATA ISSUE
   • Verify Block 4 data fetched successfully
   • Check if data covers {BACKTEST_DAYS} days

SUGGESTIONS:
• Try increasing Exit Time or adjusting TP/SL values
• Check if entry hour ({ENTRY_HOUR}) matches trading session times
• Re-run Block 4 to fetch fresh data
• Adjust BACKTEST_DAYS to ensure data coverage
""")
        print("═"*140 + "\n")
    
    backtest_trade_log = None


                                             📊 PROFESSIONAL TRADE ANALYSIS - TRADINGVIEW STYLE                                              

╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║📊 PERFORMANCE SUMMARY                                                                                                                      ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

🎯 CAPITAL & RETURNS:                                                   💰 PROFIT METRICS:
   Starting Capital:    $  100,000.00                                     Largest Win:         $    1,308.50
   Final Capital:       $   80,359.00                                     Average Win:         $      400.23
   Net Profit/Loss:     $  -19,641.00                                     Largest Loss:        $   -5,963.00
   Return on Cap

In [13]:

# ═══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC: ANALYZE ACTUAL TRADES AND ENTRY TIMES
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("🔍 TRADE ANALYSIS - CHECKING ENTRY TIMES AND DATES".center(80))
print("="*80 + "\n")

if len(all_trades) > 0:
    trades_df = pd.DataFrame(all_trades)
    
    print(f"📊 TRADE COUNT: {len(trades_df)} trades\n")
    
    # Convert entry times to IST and analyze
    trades_df['entry_datetime_ist'] = pd.to_datetime(trades_df['entry_time'])
    trades_df['entry_date_ist'] = trades_df['entry_datetime_ist'].dt.date
    trades_df['entry_hour_ist'] = trades_df['entry_datetime_ist'].dt.hour
    trades_df['entry_min_ist'] = trades_df['entry_datetime_ist'].dt.minute
    
    print(f"Entry Time Range:")
    print(f"   First Trade:   {trades_df['entry_time'].iloc[0]}")
    print(f"   Last Trade:    {trades_df['entry_time'].iloc[-1]}")
    print()
    
    # Check entry hour distribution
    entry_hour_dist = trades_df['entry_hour_ist'].value_counts().sort_index()
    print(f"Entry Hours Distribution (IST):")
    for hour in sorted(entry_hour_dist.index):
        count = entry_hour_dist[hour]
        pct = (count / len(trades_df)) * 100
        print(f"   {hour:02d}:00 IST:  {count:>3} trades ({pct:>5.1f}%)")
    print()
    
    # Unique dates
    unique_dates = trades_df['entry_date_ist'].nunique()
    print(f"Unique Dates with Trades: {unique_dates}")
    print(f"Expected (180 days):      180")
    print(f"Difference:               {180 - unique_dates}")
    print()
    
    # Sample entries at main entry hour
    if len(entry_hour_dist) > 0:
        main_hour = entry_hour_dist.idxmax()
        main_hour_trades = trades_df[trades_df['entry_hour_ist'] == main_hour]
        
        print(f"📍 Analysis of {main_hour:02d}:00 IST Entries ({len(main_hour_trades)} trades):")
        print(f"   Dates Covered:  {main_hour_trades['entry_date_ist'].nunique()}")
        print(f"   Date Range:     {main_hour_trades['entry_date_ist'].min()} to {main_hour_trades['entry_date_ist'].max()}")
        
        # Show first 5 entries
        print(f"\n   First 5 entries at {main_hour:02d}:00 IST:")
        for idx, (i, trade) in enumerate(main_hour_trades.head(5).iterrows(), 1):
            print(f"      {idx}. {trade['entry_date_ist']} {trade['entry_time']} | Price: ${trade['entry_price']:,.2f} | P&L: ${trade['pnl']:,.2f}")
else:
    print("❌ No trades recorded")




               🔍 TRADE ANALYSIS - CHECKING ENTRY TIMES AND DATES                

📊 TRADE COUNT: 133 trades

Entry Time Range:
   First Trade:   2025-11-14 05:30:00
   Last Trade:    2026-03-26 05:30:00

Entry Hours Distribution (IST):
   05:00 IST:  133 trades (100.0%)

Unique Dates with Trades: 133
Expected (180 days):      180
Difference:               47

📍 Analysis of 05:00 IST Entries (133 trades):
   Dates Covered:  133
   Date Range:     2025-11-14 to 2026-03-26

   First 5 entries at 05:00 IST:
      1. 2025-11-14 2025-11-14 05:30:00 | Price: $99,497.00 | P&L: $-47.50
      2. 2025-11-15 2025-11-15 05:30:00 | Price: $94,798.00 | P&L: $47.00
      3. 2025-11-16 2025-11-16 05:30:00 | Price: $95,590.00 | P&L: $-8.00
      4. 2025-11-17 2025-11-17 05:30:00 | Price: $94,211.50 | P&L: $18.50
      5. 2025-11-18 2025-11-18 05:30:00 | Price: $92,130.50 | P&L: $-17.00


---
## BLOCK 7️⃣ : INTERACTIVE CANDLESTICK CHARTS WITH TRADE SIGNALS

In [ ]:
---
## ANALYSIS 🔍 : SCALPING 300 POINTS - OPTIMAL ENTRY TIME & PARAMETERS

In [14]:

# ═══════════════════════════════════════════════════════════════════════════════
# SCALPING ANALYSIS: Optimize for 300 Points Daily Profit
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*140)
print("🎯 SCALPING OPTIMIZATION ANALYSIS - 300 POINTS TARGET".center(140))
print("="*140 + "\n")

if backtest_trade_log is not None and len(backtest_trade_log) > 0:
    trade_analysis_df = backtest_trade_log.copy()
    
    # Extract hour from entry time for analysis
    trade_analysis_df['Entry_DateTime'] = pd.to_datetime(trade_analysis_df['Entry Time'])
    trade_analysis_df['Entry_Hour'] = trade_analysis_df['Entry_DateTime'].dt.hour
    trade_analysis_df['Entry_Date'] = trade_analysis_df['Entry_DateTime'].dt.date
    trade_analysis_df['Trade_Duration_Hours'] = (pd.to_datetime(trade_analysis_df['Exit Time']) - 
                                                   pd.to_datetime(trade_analysis_df['Entry Time'])).dt.total_seconds() / 3600
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # SECTION 1: HOURLY ANALYSIS
    # ═════════════════════════════════════════════════════════════════════════════════
    
    print("╔" + "═"*138 + "╗")
    print("║" + "⏰ HOURLY ENTRY ANALYSIS - BEST TIMES FOR 300+ POINT SCALPS".ljust(139) + "║")
    print("╚" + "═"*138 + "╝\n")
    
    hourly_stats = trade_analysis_df.groupby('Entry_Hour').agg({
        'Net P&L (USD)': ['count', 'mean', 'sum', 'std'],
        'Net P&L (%)': 'mean',
        'Trade_Duration_Hours': 'mean',
        'R:R Ratio': 'mean'
    }).round(2)
    
    hourly_summary = pd.DataFrame()
    for hour in sorted(trade_analysis_df['Entry_Hour'].unique()):
        hour_trades = trade_analysis_df[trade_analysis_df['Entry_Hour'] == hour]
        win_count = (hour_trades['Net P&L (USD)'] > 0).sum()
        total_count = len(hour_trades)
        win_pct = (win_count / total_count * 100) if total_count > 0 else 0
        avg_pnl = hour_trades['Net P&L (USD)'].mean()
        total_pnl = hour_trades['Net P&L (USD)'].sum()
        scalp_300_count = (hour_trades['Net P&L (USD)'] >= 300).sum()
        
        hourly_summary = pd.concat([hourly_summary, pd.DataFrame({
            'Entry Hour (IST)': [f"{hour:02d}:00"],
            'Trades Count': [total_count],
            'Win Rate %': [f"{win_pct:.1f}%"],
            'Avg P&L': [f"${avg_pnl:,.2f}"],
            'Total P&L': [f"${total_pnl:,.2f}"],
            'Trades with 300+ pts': [scalp_300_count],
            'Avg Duration (hrs)': [f"{hour_trades['Trade_Duration_Hours'].mean():.2f}"]
        })], ignore_index=True)
    
    print(hourly_summary.to_string(index=False))
    print()
    
    # Find best hour for 300 points
    best_hour_trades = trade_analysis_df.copy()
    best_hour_trades['scalp_300'] = best_hour_trades['Net P&L (USD)'] >= 300
    hourly_scalp_count = best_hour_trades.groupby('Entry_Hour')['scalp_300'].sum()
    best_scalp_hour = hourly_scalp_count.idxmax() if len(hourly_scalp_count) > 0 else 0
    best_scalp_count = hourly_scalp_count.max() if len(hourly_scalp_count) > 0 else 0
    
    print("\n" + "─"*140)
    print(f"🏆 BEST ENTRY HOUR FOR 300+ POINT SCALPS: {best_scalp_hour:02d}:00 IST")
    print(f"   → Achieved 300+ points: {int(best_scalp_count)} times out of {len(trade_analysis_df[trade_analysis_df['Entry_Hour'] == best_scalp_hour])} trades")
    print(f"   → Success Rate: {(best_scalp_count / len(trade_analysis_df[trade_analysis_df['Entry_Hour'] == best_scalp_hour]) * 100):.1f}%")
    print("─"*140 + "\n")
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # SECTION 2: VOLATILITY & PRICE MOVEMENT ANALYSIS
    # ═════════════════════════════════════════════════════════════════════════════════
    
    print("\n╔" + "═"*138 + "╗")
    print("║" + "📊 VOLATILITY ANALYSIS - PRICE MOVEMENT PATTERNS".ljust(139) + "║")
    print("╚" + "═"*138 + "╝\n")
    
    print(f"Average Daily Price Range:")
    print(f"   Max Price Range per Trade:  ${trade_analysis_df['Price Range'].max():>10,.2f}")
    print(f"   Avg Price Range per Trade:  ${trade_analysis_df['Price Range'].mean():>10,.2f}")
    print(f"   Min Price Range per Trade:  ${trade_analysis_df['Price Range'].min():>10,.2f}")
    print()
    
    print(f"Favorable & Adverse Moves:")
    print(f"   Avg Favorable Move:         ${trade_analysis_df['Favorable'].mean():>10,.2f} (max profit potential)")
    print(f"   Avg Adverse Move:           ${trade_analysis_df['Adverse'].mean():>10,.2f} (max loss potential)")
    print(f"   Avg Risk:Reward Ratio:      {trade_analysis_df['R:R Ratio'].mean():>14.2f}x")
    print()
    
    # Analyze 300-point achievability
    trades_300_plus = trade_analysis_df[trade_analysis_df['Net P&L (USD)'] >= 300]
    trades_500_plus = trade_analysis_df[trade_analysis_df['Net P&L (USD)'] >= 500]
    
    print(f"Scalping Potential:")
    print(f"   Trades that reached 300+ points: {len(trades_300_plus):>5} out of {len(trade_analysis_df):>5} ({(len(trades_300_plus)/len(trade_analysis_df)*100):>5.1f}%)")
    print(f"   Trades that reached 500+ points: {len(trades_500_plus):>5} out of {len(trade_analysis_df):>5} ({(len(trades_500_plus)/len(trade_analysis_df)*100):>5.1f}%)")
    
    if len(trades_300_plus) > 0:
        print(f"\n   300+ Point Trades Statistics:")
        print(f"   • Average P&L:              ${trades_300_plus['Net P&L (USD)'].mean():>10,.2f}")
        print(f"   • Max P&L:                  ${trades_300_plus['Net P&L (USD)'].max():>10,.2f}")
        print(f"   • Min P&L:                  ${trades_300_plus['Net P&L (USD)'].min():>10,.2f}")
        print(f"   • Avg Duration:             {trades_300_plus['Trade_Duration_Hours'].mean():>10.2f} hours")
        print(f"   • Avg Favorable Move:       ${trades_300_plus['Favorable'].mean():>10,.2f}")
    
    print()
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # SECTION 3: OPTIMAL PARAMETERS FOR 300-POINT SCALPING
    # ═════════════════════════════════════════════════════════════════════════════════
    
    print("\n╔" + "═"*138 + "╗")
    print("║" + "⚙️  RECOMMENDED PARAMETERS FOR 300-POINT SCALPING".ljust(139) + "║")
    print("╚" + "═"*138 + "╝\n")
    
    # Calculate TP needed for 300 points
    avg_entry_price = (trade_analysis_df['Entry Price'].mean() + trade_analysis_df['Exit Price'].mean()) / 2
    points_target = 300.0
    position_size = POSITION_SIZE
    
    # TP in $ terms: 300 points / position size = TP target
    tp_target_price_range = points_target / position_size
    tp_percent_needed = (tp_target_price_range / avg_entry_price) * 100
    tp_points_target = points_target  # 300 points = $300 per 1 BTC
    
    print(f"Current Average Entry Price: ${avg_entry_price:,.2f}")
    print(f"Target Profit: {points_target:,.0f} points per trade\n")
    
    print(f"📌 RECOMMENDED CONFIGURATION FOR 300-POINT SCALPING:\n")
    print(f"┌─ ENTRY TIME ─────────────────────────────────────────────────────────────────┐")
    print(f"│  Entry Hour:     {best_scalp_hour:02d}:00 IST (Best hour for 300+ point trades)")
    print(f"│  Entry Minute:   00")
    print(f"│  Reasoning:      Highest success rate for 300+ point scalps")
    print(f"└───────────────────────────────────────────────────────────────────────────────┘\n")
    
    print(f"┌─ TAKE PROFIT SETTINGS ────────────────────────────────────────────────────────┐")
    print(f"│  TP_MODE:                  POINTS")
    print(f"│  TAKE_PROFIT_POINTS:       300.0      (Scalp exactly 300 points)")
    print(f"│")
    print(f"│  Alternative (Percentage):")
    print(f"│  TP_MODE:                  PERCENTAGE")
    print(f"│  TAKE_PROFIT_PERCENT:      {tp_percent_needed:.2f}%   (Equivalent to 300 points)")
    print(f"└───────────────────────────────────────────────────────────────────────────────┘\n")
    
    # Calculate SL needed
    avg_adverse = trade_analysis_df['Adverse'].mean()
    sl_recommended = avg_adverse * 0.8  # Use 80% of average adverse move
    sl_percent_recommended = (sl_recommended / avg_entry_price) * 100
    
    print(f"┌─ STOP LOSS SETTINGS ──────────────────────────────────────────────────────────┐")
    print(f"│  SL_MODE:                  POINTS")
    print(f"│  STOP_LOSS_POINTS:         {sl_recommended:,.0f}    (Conservative based on volatility)")
    print(f"│")
    print(f"│  Alternative (Percentage):")
    print(f"│  SL_MODE:                  PERCENTAGE")
    print(f"│  STOP_LOSS_PERCENT:        {sl_percent_recommended:.2f}%")
    print(f"└───────────────────────────────────────────────────────────────────────────────┘\n")
    
    print(f"┌─ TRADING MODE & POSITION ────────────────────────────────────────────────────┐")
    print(f"│  TRADING_MODE:             LONG_ONLY  (based on historical performance)")
    print(f"│  POSITION_SIZE:            {POSITION_SIZE} BTC")
    print(f"│  EXIT_HOUR:                23")
    print(f"│  EXIT_MIN:                 58         (Just before market close)")
    print(f"└───────────────────────────────────────────────────────────────────────────────┘\n")
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # SECTION 4: DAILY TARGET ANALYSIS
    # ═════════════════════════════════════════════════════════════════════════════════
    
    print("\n╔" + "═"*138 + "╗")
    print("║" + "💰 DAILY TARGET FEASIBILITY - CAN YOU SCALP 300 POINTS EVERY DAY?".ljust(139) + "║")
    print("╚" + "═"*138 + "╝\n")
    
    daily_stats = trade_analysis_df.groupby('Entry_Date').agg({
        'Net P&L (USD)': ['count', 'sum', 'mean']
    }).round(2)
    
    daily_pnl = trade_analysis_df.groupby('Entry_Date')['Net P&L (USD)'].sum()
    days_300_plus = (daily_pnl >= 300).sum()
    days_total = len(daily_pnl)
    
    print(f"Trading Days Analysis (from Entry Date):")
    print(f"   Total Trading Days:        {days_total}")
    print(f"   Days with 300+ points:     {days_300_plus}")
    print(f"   Success Rate:              {(days_300_plus/days_total*100):.1f}%")
    print(f"   Trades per Day (average):  {trade_analysis_df.groupby('Entry_Date').size().mean():.1f}")
    print(f"   Max P&L in a Day:          ${daily_pnl.max():,.2f}")
    print(f"   Min P&L in a Day:          ${daily_pnl.min():,.2f}")
    print(f"   Avg P&L per Day:           ${daily_pnl.mean():,.2f}\n")
    
    if days_300_plus / days_total >= 0.7:
        print("✅ ASSESSMENT: 300 POINTS IS ACHIEVABLE DAILY!")
        print(f"   Success Rate: {(days_300_plus/days_total*100):.1f}% - GOOD (>70%)")
    elif days_300_plus / days_total >= 0.5:
        print("⚠️  ASSESSMENT: 300 POINTS IS MODERATELY ACHIEVABLE")
        print(f"   Success Rate: {(days_300_plus/days_total*100):.1f}% - FAIR (50-70%, needs optimization)")
    else:
        print("❌ ASSESSMENT: 300 POINTS IS CHALLENGING")
        print(f"   Success Rate: {(days_300_plus/days_total*100):.1f}% - LOW (<50%, increase TP or adjust strategy)")
    
    print()
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # SECTION 5: ENHANCED TRADE LOG FOR GOOGLE SHEETS
    # ═════════════════════════════════════════════════════════════════════════════════
    
    print("\n╔" + "═"*138 + "╗")
    print("║" + "📥 CREATING ENHANCED TRADE LOG FOR GOOGLE SHEETS".ljust(139) + "║")
    print("╚" + "═"*138 + "╝\n")
    
    # Create enhanced export with additional analysis columns
    enhanced_log = trade_analysis_df.copy()
    
    # Add columns for Google Sheets analysis
    enhanced_log['Trend'] = enhanced_log['Net P&L (USD)'].apply(lambda x: '✓ WIN' if x > 0 else ('✗ LOSS' if x < 0 else '- B/E'))
    enhanced_log['Points'] = enhanced_log['Net P&L (USD)'].apply(lambda x: f"{x:,.0f}")
    enhanced_log['300pt_Target'] = enhanced_log['Net P&L (USD)'].apply(lambda x: '✓ ACHIEVED' if x >= 300 else f"Need {300-x:,.0f}")
    enhanced_log['Entry_Date_Only'] = enhanced_log['Entry_DateTime'].dt.date
    enhanced_log['Entry_IST_Time'] = enhanced_log['Entry_DateTime'].dt.strftime('%H:%M')
    enhanced_log['Day_of_Week'] = enhanced_log['Entry_DateTime'].dt.day_name()
    enhanced_log['Scalp_Quality'] = enhanced_log.apply(
        lambda row: 'Excellent' if row['Net P&L (USD)'] >= 500 
        else ('Good' if row['Net P&L (USD)'] >= 300 
        else ('Fair' if row['Net P&L (USD)'] >= 100 
        else 'Poor')), axis=1
    )
    
    # Create the export dataframe with meaningful columns
    google_sheets_export = enhanced_log[[
        'Trade #', 'Entry_Date_Only', 'Day_of_Week', 'Entry_IST_Time', 'Entry_Hour',
        'Entry Price', 'Exit Price', 'Type', 'Position Size',
        'Exit Reason', 'Points', 'Net P&L (%)', 'Trend', 'Scalp_Quality', '300pt_Target',
        'Favorable', 'Adverse', 'R:R Ratio', 'Trade_Duration_Hours', 'Cumulative P&L'
    ]].copy()
    
    google_sheets_export.columns = [
        'Trade#', 'Date', 'Day', 'Time(IST)', 'Hour', 'Entry$', 'Exit$', 'Type', 'Size',
        'ExitReason', 'Points', 'PnL%', 'Result', 'Quality', '300pt_Status',
        'FavorableMove', 'AdverseMove', 'R:R', 'Duration(hrs)', 'CumulativeP&L'
    ]
    
    # Export to CSV with better formatting
    enhanced_csv = 'day_trading_log_enhanced.csv'
    google_sheets_export.to_csv(enhanced_csv, index=False)
    
    print(f"✅ Enhanced Trade Log Created: {enhanced_csv}")
    print(f"📊 Total Trades: {len(google_sheets_export)}")
    print(f"📈 Columns: {len(google_sheets_export.columns)}")
    print(f"\nColumns included for Google Sheets analysis:")
    for col in google_sheets_export.columns:
        print(f"   • {col}")
    
    print(f"\n💾 File ready for import to Google Sheets!")
    print(f"   Download: {enhanced_csv}")
    print()
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # SECTION 6: QUICK PARAMETER RECOMMENDATION SUMMARY
    # ═════════════════════════════════════════════════════════════════════════════════
    
    print("\n" + "╔" + "═"*138 + "╗")
    print("║" + "🎯 FINAL RECOMMENDATION - COPY THIS TO BLOCK 2".ljust(139) + "║")
    print("╚" + "═"*138 + "╝\n")
    
    recommended_sl_points = int(sl_recommended)
    
    print(f"""
# ════════════════════ BLOCK 2: COPY THESE SETTINGS ════════════════════
# OPTIMAL PARAMETERS FOR 300-POINT DAILY SCALPING

TRADING_MODE = "LONG_ONLY"          # Best for this volatility

ENTRY_HOUR = {best_scalp_hour}                      # OPTIMAL ENTRY TIME
ENTRY_MIN = 0
EXIT_HOUR = 23
EXIT_MIN = 58

# TAKE PROFIT - Points Mode (RECOMMENDED)
USE_TAKE_PROFIT = True
TP_MODE = "POINTS"                   # Use fixed points
TAKE_PROFIT_POINTS = 300.0           # Target exactly 300 points
TAKE_PROFIT_PERCENT = 0.4            # Fallback

# STOP LOSS - Conservative
USE_STOP_LOSS = True
SL_MODE = "POINTS"                   # Fixed stop loss
STOP_LOSS_POINTS = {recommended_sl_points}                # Based on analysis
STOP_LOSS_PERCENT = 1.0              # Fallback

INITIAL_CAPITAL = 100000
POSITION_SIZE = 1.0
COMMISSION_PERCENT = 0.05
SLIPPAGE_USD = 2.0
BACKTEST_DAYS = 180

# ════════════════════════════════════════════════════════════════════════
    """)
    
    print(f"\n✅ After copying parameters, run Blocks 2 → 5 → 6 to see results!")
    
    # Display sample trades from best hour
    print("\n" + "─"*140)
    print(f"📋 Sample Trades from Best Hour ({best_scalp_hour:02d}:00 IST) - Last 5 trades:")
    print("─"*140)
    
    best_hour_sample = trade_analysis_df[trade_analysis_df['Entry_Hour'] == best_scalp_hour].tail(5)
    for idx, (i, trade) in enumerate(best_hour_sample.iterrows(), 1):
        status = "✓" if trade['Net P&L (USD)'] >= 300 else "✗"
        print(f"{status} Trade #{int(trade['Trade #']):3d} | {trade['Type']:<5} | Entry: ${trade['Entry Price']:>10,.2f} | Exit: ${trade['Exit Price']:>10,.2f} | P&L: ${trade['Net P&L (USD)']:>10,.2f} | {trade['Exit Reason']}")
    
    print("\n" + "="*140 + "\n")

else:
    print("⚠️  No trade data available. Please run Blocks 1-6 first to generate trades.")



                                            🎯 SCALPING OPTIMIZATION ANALYSIS - 300 POINTS TARGET                                            

╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║⏰ HOURLY ENTRY ANALYSIS - BEST TIMES FOR 300+ POINT SCALPS                                                                                 ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

Entry Hour (IST)  Trades Count Win Rate % Avg P&L  Total P&L  Trades with 300+ pts Avg Duration (hrs)
           05:00           133      46.6%  $-7.68 $-1,021.00                     1               0.02


────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
🏆 BEST ENTRY HOUR FOR 300+ POINT SCALPS: 05:00 IST
   → Achieved 300+ points: 1 times

In [16]:
# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 7: INTERACTIVE CANDLESTICK CHARTS WITH INDICATORS & TRADE SIGNALS
# ═══════════════════════════════════════════════════════════════════════════════

import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("\n" + "="*80)
print("📊 GENERATING INTERACTIVE CHARTS WITH TRADE SIGNALS".center(80))
print("="*80 + "\n")

if backtest_df is not None:
    # Calculate indicators for charting
    chart_df = backtest_df.copy()
    chart_df = calculate_moving_averages(chart_df)
    chart_df = calculate_bollinger_bands(chart_df)
    chart_df = calculate_atr(chart_df)
    chart_df = calculate_rsi(chart_df)
    
    # Use last 5000 candles for chart clarity
    chart_data = chart_df.tail(5000).reset_index(drop=True)
    
    print(f"📈 Preparing chart with {len(chart_data)} candles...\n")
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # PROCESS TRADE SIGNALS
    # ═════════════════════════════════════════════════════════════════════════════════
    
    buy_signals = []
    sell_signals = []
    
    if backtest_trade_log is not None and len(backtest_trade_log) > 0:
        # Extract buy (entry) and sell (exit) signals from actual trades
        for idx, trade in backtest_trade_log.iterrows():
            entry_time = pd.to_datetime(trade['Entry Time'])
            exit_time = pd.to_datetime(trade['Exit Time'])
            
            # Find matching timestamps in chart data
            entry_matches = chart_data[chart_data['timestamp'].dt.floor('min') == entry_time.floor('min')]
            exit_matches = chart_data[chart_data['timestamp'].dt.floor('min') == exit_time.floor('min')]
            
            if not entry_matches.empty:
                entry_row = entry_matches.iloc[0]
                if trade['Type'] == 'LONG':
                    buy_signals.append({
                        'time': entry_row['timestamp'],
                        'price': trade['Entry Price'],
                        'type': 'BUY',
                        'trade_num': trade['Trade #']
                    })
                else:
                    sell_signals.append({
                        'time': entry_row['timestamp'],
                        'price': trade['Entry Price'],
                        'type': 'SELL',
                        'trade_num': trade['Trade #']
                    })
            
            if not exit_matches.empty:
                exit_row = exit_matches.iloc[0]
                if trade['Type'] == 'LONG':
                    sell_signals.append({
                        'time': exit_row['timestamp'],
                        'price': trade['Exit Price'],
                        'type': 'EXIT_LONG',
                        'trade_num': trade['Trade #'],
                        'reason': trade['Exit Reason']
                    })
                else:
                    buy_signals.append({
                        'time': exit_row['timestamp'],
                        'price': trade['Exit Price'],
                        'type': 'COVER_SHORT',
                        'trade_num': trade['Trade #'],
                        'reason': trade['Exit Reason']
                    })
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # CREATE MULTI-SUBPLOT FIGURE
    # ═════════════════════════════════════════════════════════════════════════════════
    
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.15,
        row_heights=[0.7, 0.3],
        specs=[[{"secondary_y": True}], [{"secondary_y": False}]]
    )
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # CANDLESTICK CHART
    # ═════════════════════════════════════════════════════════════════════════════════
    
    fig.add_trace(
        go.Candlestick(
            x=chart_data['timestamp'],
            open=chart_data['open'],
            high=chart_data['high'],
            low=chart_data['low'],
            close=chart_data['close'],
            name='OHLC',
            showlegend=True,
            hovertemplate='<b>%{x}</b><br>O:$%{open:.2f} H:$%{high:.2f} L:$%{low:.2f} C:$%{close:.2f}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # TECHNICAL INDICATORS
    # ═════════════════════════════════════════════════════════════════════════════════
    
    # Moving Averages
    for period, color in [(9, 'orange'), (21, 'purple'), (50, 'gray')]:
        fig.add_trace(
            go.Scatter(
                x=chart_data['timestamp'],
                y=chart_data[f'SMA_{period}'],
                name=f'SMA {period}',
                line=dict(color=color, width=1, dash='dot'),
                hovertemplate=f'SMA {period}: $%{{y:.2f}}<extra></extra>'
            ),
            row=1, col=1
        )
    
    # Bollinger Bands
    fig.add_trace(
        go.Scatter(
            x=chart_data['timestamp'],
            y=chart_data['BB_UPPER'],
            name='BB Upper',
            line=dict(color='rgba(255, 100, 100, 0.5)', width=1, dash='dash'),
            hovertemplate='BB Upper: $%{y:.2f}<extra></extra>'
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=chart_data['timestamp'],
            y=chart_data['BB_LOWER'],
            name='BB Lower',
            line=dict(color='rgba(100, 255, 100, 0.5)', width=1, dash='dash'),
            fill='tonexty',
            fillcolor='rgba(200, 200, 200, 0.1)',
            hovertemplate='BB Lower: $%{y:.2f}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # BUY SIGNALS (Trade Entries & COVERs)
    # ═════════════════════════════════════════════════════════════════════════════════
    
    if buy_signals:
        buy_times = [s['time'] for s in buy_signals]
        buy_prices = [s['price'] for s in buy_signals]
        buy_text = [f"{'LONG ENTRY' if s['type'] == 'BUY' else 'COVER SHORT'}<br>Trade #{s['trade_num']}<br>${s['price']:.2f}" 
                   for s in buy_signals]
        
        fig.add_trace(
            go.Scatter(
                x=buy_times,
                y=buy_prices,
                mode='markers',
                name='Buy Signals',
                marker=dict(
                    size=12,
                    color='green',
                    symbol='triangle-up',
                    line=dict(color='white', width=2)
                ),
                text=buy_text,
                hovertemplate='<b>%{text}</b><extra></extra>',
                showlegend=True
            ),
            row=1, col=1
        )
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # SELL SIGNALS (Trade Exits & SHORTs)
    # ═════════════════════════════════════════════════════════════════════════════════
    
    if sell_signals:
        sell_times = [s['time'] for s in sell_signals]
        sell_prices = [s['price'] for s in sell_signals]
        sell_text = [f"{'SHORT ENTRY' if s['type'] == 'SELL' else 'EXIT LONG'}<br>Trade #{s['trade_num']}<br>${s['price']:.2f}<br>{s.get('reason', '')}" 
                    for s in sell_signals]
        
        fig.add_trace(
            go.Scatter(
                x=sell_times,
                y=sell_prices,
                mode='markers',
                name='Sell Signals',
                marker=dict(
                    size=12,
                    color='red',
                    symbol='triangle-down',
                    line=dict(color='white', width=2)
                ),
                text=sell_text,
                hovertemplate='<b>%{text}</b><extra></extra>',
                showlegend=True
            ),
            row=1, col=1
        )
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # VOLUME CHART
    # ═════════════════════════════════════════════════════════════════════════════════
    
    colors = ['red' if chart_data['close'].iloc[i] < chart_data['open'].iloc[i] else 'green' 
              for i in range(len(chart_data))]
    
    fig.add_trace(
        go.Bar(
            x=chart_data['timestamp'],
            y=chart_data['volume'],
            name='Volume',
            marker=dict(color=colors),
            hovertemplate='Volume: %{y:.2f} BTC<extra></extra>'
        ),
        row=2, col=1
    )
    
    # ═════════════════════════════════════════════════════════════════════════════════
    # UPDATE LAYOUT FOR INTERACTIVITY
    # ═════════════════════════════════════════════════════════════════════════════════
    
    fig.update_layout(
        title=f'Day Trading Strategy - {SYMBOL} {TIMEFRAME} | Mode: {TRADING_MODE} | Entry: {ENTRY_HOUR:02d}:{ENTRY_MIN:02d} IST',
        template='plotly_dark',
        height=900,
        hovermode='x unified',
        xaxis=dict(
            rangeslider=dict(visible=False),
            type='date',
            rangeselector=dict(
                buttons=list([
                    dict(count=1, label='1D', step='day', stepmode='backward'),
                    dict(count=7, label='1W', step='day', stepmode='backward'),
                    dict(count=14, label='2W', step='day', stepmode='backward'),
                    dict(count=1, label='1M', step='month', stepmode='backward'),
                    dict(count=3, label='3M', step='month', stepmode='backward'),
                    dict(step='all', label='All')
                ])
            )
        ),
        xaxis2=dict(rangeslider=dict(visible=False))
    )
    
    fig.update_yaxes(title_text='Price (USD)', row=1, col=1)
    fig.update_yaxes(title_text='Volume (BTC)', row=2, col=1)
    
    # Save chart
    chart_file = 'day_trading_chart.html'
    fig.write_html(chart_file)
    
    # Open chart in browser
    import webbrowser
    from IPython.display import HTML, display
    
    chart_path = os.path.abspath(chart_file)
    webbrowser.open('file://' + chart_path)
    
    # Display clickable link
    display(HTML(f'''
    <div style="background-color: #1e1e1e; padding: 15px; border-radius: 5px; margin: 10px 0;">
        <p style="color: #00ff00; font-size: 14px;">📊 <b>Interactive Chart Ready!</b></p>
        <a href="file:///{chart_path.replace(chr(92), '/')}" target="_blank" 
           style="color: #00d4ff; font-size: 16px; text-decoration: none; font-weight: bold;">
           ➤ Click here to open the interactive chart with trade signals
        </a>
        <p style="color: #888; font-size: 12px; margin-top: 8px;">Or open: {chart_path}</p>
    </div>
    ''')
    )
    
    print(f"\n╔" + "═"*78 + "╗")
    print("║" + "✅ INTERACTIVE CHART WITH TRADE SIGNALS CREATED".center(78) + "║")
    print("╚" + "═"*78 + "╝")
    print(f"\n📊 Chart Features:")
    print(f"   ✓ Candlestick OHLC")
    print(f"   ✓ Moving Averages (SMA 9, 21, 50)")
    print(f"   ✓ Bollinger Bands")
    print(f"   ✓ BUY SIGNALS (Green Triangles Up)")
    print(f"   ✓ SELL SIGNALS (Red Triangles Down)")
    print(f"   ✓ Volume Bars (Color-coded)")
    print(f"   ✓ Time Range Selector (1D, 1W, 2W, 1M, 3M, All)")
    print(f"   ✓ Scrollable & Zoomable Axes")
    print(f"   ✓ Hover Information with Trade Details\n")
    print(f"   Buy Signals Found: {len(buy_signals)}")
    print(f"   Sell Signals Found: {len(sell_signals)}")
    print(f"\n💾 Chart saved to: {chart_file}")
    print(f"\n🚀 Open in browser to interact with the chart!\n")
    
    # Display the chart
    fig.show()
else:
    print("❌ No data available for charting")


               📊 GENERATING INTERACTIVE CHARTS WITH TRADE SIGNALS               

📈 Preparing chart with 5000 candles...




╔══════════════════════════════════════════════════════════════════════════════╗
║                ✅ INTERACTIVE CHART WITH TRADE SIGNALS CREATED                ║
╚══════════════════════════════════════════════════════════════════════════════╝

📊 Chart Features:
   ✓ Candlestick OHLC
   ✓ Moving Averages (SMA 9, 21, 50)
   ✓ Bollinger Bands
   ✓ BUY SIGNALS (Green Triangles Up)
   ✓ SELL SIGNALS (Red Triangles Down)
   ✓ Volume Bars (Color-coded)
   ✓ Time Range Selector (1D, 1W, 2W, 1M, 3M, All)
   ✓ Scrollable & Zoomable Axes
   ✓ Hover Information with Trade Details

   Buy Signals Found: 0
   Sell Signals Found: 0

💾 Chart saved to: day_trading_chart.html

🚀 Open in browser to interact with the chart!

